In [20]:
import sqlite3
import pandas as pd
import os



os.makedirs('chess_db', exist_ok=True)
    
games_url = "https://drive.google.com/uc?export=download&id=1eR3NZtwIC6ECN3vhtrynqmx8okG0twA7"
players_url = "https://drive.google.com/uc?export=download&id=1wCSAkGagMzWiToedLC3ZGo_lGf_laF-k"
    
conn = sqlite3.connect('chess_db/games.db')
conn.execute('PRAGMA foreign_keys = ON')

df1 = pd.read_csv(games_url)
df2 = pd.read_csv(players_url)
    
    

In [21]:
df1 = df1.drop_duplicates(subset = "game_id")
df2 = df2.drop_duplicates(subset = "username")

df1.to_sql('games', conn, if_exists='replace', index=False)
df2.to_sql('players', conn, if_exists='replace', index=False)


215

In [22]:

openings_cols = ['opening_code', 'opening_fullname', 'opening_shortname', 
                 'opening_moves', 'opening_response', 'opening_variation']
df_openings = df1[openings_cols].drop_duplicates('opening_code')
df_openings.to_sql('openings', conn, if_exists='replace', index=False)
openings_cols_new = ['opening_fullname', 'opening_shortname', 
                 'opening_moves', 'opening_response', 'opening_variation']
df_games = df1.drop(columns=openings_cols_new)
df_games.to_sql('games', conn, if_exists='replace', index=False)



20058

In [23]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_white_id ON games(white_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_black_id ON games(black_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_opening_code ON games(opening_code)")

In [25]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE white_id = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_white_id (white_id=?)')]


In [26]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE black_id = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_black_id (black_id=?)')]


In [27]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE opening_code = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_opening_code (opening_code=?)')]
